In [1]:
!pip install SALib numpy pandas

import numpy as np
import pandas as pd

from SALib.sample.morris import sample as morris_sample
from SALib.analyze.morris import analyze as morris_analyze

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.1/780.1 kB 10.8 MB/s eta 0:00:00


In [5]:



# ----------------------------
# 1) Define the Morris problem
# ----------------------------
problem = {
    "num_vars": 5,
    "names": ["carbon_tax", "gas_var_cost", "coal_var_cost", "demand", "ccg_eff"],
    "bounds": [
        [0.0, 200.0],   # carbon_tax (example range)
        [0.5, 2.0],     # gas_var_cost multiplier
        [0.5, 2.0],     # coal_var_cost multiplier
        [0.8, 1.2],     # demand multiplier
        [0.45, 0.60],   # CCG efficiency
    ],
}

# Morris design choices (typical starting point)
N = 300               # number of trajectories (increase for stability)
num_levels = 8        # grid levels
optimal_trajectories = 50  # improves space-filling of trajectories


# ---------------------------------------
# 2) Generate the Morris sample (inputs X)
# ---------------------------------------
X = morris_sample(
    problem,
    N=N,
    num_levels=num_levels,
    optimal_trajectories=optimal_trajectories,
)

# X shape: [N*(num_vars+1), num_vars]
print("Morris samples:", X.shape)


# ----------------------------------------
# 3) Define your model runner: y = f(x)
# ----------------------------------------
def run_model(x_row: np.ndarray) -> float:
    """
    Replace this stub with your ESOM run:
      - write parameters into config/input files
      - run the model (e.g., OSeMOSYS / your pipeline)
      - parse output metric (objective cost, emissions, RES share, ...)
    Must return ONE scalar output for this call.
    """
    carbon_tax, gas_var_cost, coal_var_cost, demand, ccg_eff = x_row

    # ---- EXAMPLE DUMMY FUNCTION (DELETE) ----
    # This is just a placeholder so the script runs.
    # Replace with your actual model evaluation.
    y = (
        1000 * demand
        + 200 * gas_var_cost
        + 50 * carbon_tax
        - 300 * (ccg_eff - 0.45)
        + 80 * coal_var_cost
    )
    return float(y)


# Evaluate model for all samples
Y = np.array([run_model(x) for x in X], dtype=float)


# ----------------------------------------
# 4) Morris analysis: mu*, mu, sigma
# ----------------------------------------
Si = morris_analyze(
    problem,
    X,
    Y,
    conf_level=0.95,
    num_levels=num_levels,
    num_resamples=1000,   # bootstrap for CI
)

# Convert X to a DataFrame with factor names
X_df = pd.DataFrame(X, columns=problem["names"])

# Look at the first 20 sample points
print(X_df.head(20))

# Si contains: mu, mu_star, sigma, mu_star_conf
results = pd.DataFrame({
    "factor": problem["names"],
    "mu": Si["mu"],
    "mu_star": Si["mu_star"],
    "sigma": Si["sigma"],
    "mu_star_conf": Si["mu_star_conf"],
}).sort_values("mu_star", ascending=False)

print(results.to_string(index=False))


# ----------------------------------------
# 5) Save results
# ----------------------------------------
results.to_csv("morris_results.csv", index=False)
print("Saved: morris_results.csv")

Morris samples: (300, 5)
    carbon_tax  gas_var_cost  coal_var_cost    demand   ccg_eff
0   114.285714      1.142857       0.500000  0.971429  0.492857
1   114.285714      1.142857       0.500000  1.200000  0.492857
2   114.285714      2.000000       0.500000  1.200000  0.492857
3     0.000000      2.000000       0.500000  1.200000  0.492857
4     0.000000      2.000000       1.357143  1.200000  0.492857
5     0.000000      2.000000       1.357143  1.200000  0.578571
6    57.142857      2.000000       0.928571  1.028571  0.578571
7    57.142857      2.000000       0.928571  0.800000  0.578571
8    57.142857      2.000000       1.785714  0.800000  0.578571
9   171.428571      2.000000       1.785714  0.800000  0.578571
10  171.428571      2.000000       1.785714  0.800000  0.492857
11  171.428571      1.142857       1.785714  0.800000  0.492857
12   57.142857      1.571429       0.500000  0.971429  0.600000
13  171.428571      1.571429       0.500000  0.971429  0.600000
14  171.428571 

In [4]:
# Convert X to a DataFrame with factor names
X_df = pd.DataFrame(X, columns=problem["names"])

# Look at the first 20 sample points
print(X_df.head(20))

    carbon_tax  gas_var_cost  coal_var_cost    demand   ccg_eff
0    28.571429      2.000000       1.571429  0.971429  0.535714
1    28.571429      2.000000       1.571429  0.971429  0.450000
2    28.571429      2.000000       1.571429  1.200000  0.450000
3    28.571429      2.000000       0.714286  1.200000  0.450000
4    28.571429      1.142857       0.714286  1.200000  0.450000
5   142.857143      1.142857       0.714286  1.200000  0.450000
6   171.428571      0.928571       1.142857  0.800000  0.578571
7   171.428571      0.928571       2.000000  0.800000  0.578571
8   171.428571      1.785714       2.000000  0.800000  0.578571
9   171.428571      1.785714       2.000000  1.028571  0.578571
10   57.142857      1.785714       2.000000  1.028571  0.578571
11   57.142857      1.785714       2.000000  1.028571  0.492857
12   85.714286      1.571429       2.000000  1.200000  0.514286
13   85.714286      0.714286       2.000000  1.200000  0.514286
14   85.714286      0.714286       2.000